# Lecture 2: Lasso and Elastic Net Regression

### Short, simple, self-study notes

This lesson recaps Ridge regression, explains Lasso feature selection, and shows how Elastic Net combines Ridge and Lasso.

**Main idea:** Ridge makes weights smaller, Lasso can make some weights exactly zero, and Elastic Net does both.

## 1. Why do we need regularization?

A linear model can learn the training data too closely. Then it gets very high training accuracy but poor test accuracy. This is **overfitting**.

Regularization adds an extra penalty for large coefficients (feature weights):

**total cost = prediction error + regularization penalty**

The penalty is controlled by **lambda (λ)**. In scikit-learn, this setting is usually called **alpha**.

- Small alpha: a light penalty.
- Large alpha: a strong penalty and smaller coefficients.
- Too much penalty can cause underfitting.

> Important: the intercept is normally not included in the penalty.

## 2. Ridge recap: L2 regularization

Ridge adds the **squared coefficients** to the cost:

**cost = prediction error + λ × (w₁² + w₂² + ... + wₙ²)**

Ridge pulls coefficients toward zero, but usually does not make them exactly zero. So it generally keeps every feature, just with less influence.

**Interview answer:** Use Ridge mainly to reduce overfitting, especially when many features are useful or correlated with each other.

## 3. Lasso regression: L1 regularization

Lasso adds the **absolute values of the coefficients** to the cost:

**cost = prediction error + λ × (|w₁| + |w₂| + ... + |wₙ|)**

### What makes Lasso special?

Lasso can push some coefficients to **exactly 0**. A feature with a zero coefficient is switched off, so Lasso performs **feature selection** automatically.

### Simple example

Suppose a model is:

**score = 50 + 0.8 × study_hours + 0.6 × practice_questions + 0.02 × sticker_count**

If sticker_count is not useful, Lasso may change 0.02 to 0. The model then ignores that feature:

**score = 50 + 0.8 × study_hours + 0.6 × practice_questions + 0 × sticker_count**

**Interview answer:** Use Lasso when you have many features and want the model to select a smaller, more useful set.

## 4. Why does Lasso select features?

The L1 penalty uses absolute values. Its shape has a sharp corner at zero. That corner makes it easier for an optimized coefficient to land exactly on zero.

Ridge uses squares. Its smooth, round penalty usually shrinks coefficients close to zero without reaching zero.

The next diagram is only a visual explanation of the penalty shapes. It is not the complete model cost.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

coefficient = np.linspace(-3, 3, 400)
l1_penalty = np.abs(coefficient)       # Lasso: |w|
l2_penalty = coefficient ** 2          # Ridge: w²

plt.figure(figsize=(8, 4))
plt.plot(coefficient, l1_penalty, label="L1 / Lasso: |w|", linewidth=3)
plt.plot(coefficient, l2_penalty, label="L2 / Ridge: w²", linewidth=3)
plt.axvline(0, color="black", linewidth=0.8)
plt.axhline(0, color="black", linewidth=0.8)
plt.scatter([0], [0], color="red", zorder=5, label="zero coefficient")
plt.xlabel("coefficient value (w)")
plt.ylabel("penalty size")
plt.title("Penalty shapes: L1 has a sharp corner at zero")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

### How to read the diagram

- The horizontal axis is a coefficient value.
- The vertical axis is the penalty paid for that coefficient.
- L1 has a sharp point at zero, which helps some coefficients become exactly zero.
- L2 is smooth, so it usually makes coefficients small rather than exactly zero.

## 5. Elastic Net: L1 + L2

Elastic Net combines both penalties:

**cost = prediction error + λ₁ × (sum of |w|) + λ₂ × (sum of w²)**

So it can:

- reduce overfitting like Ridge;
- set some coefficients to zero like Lasso;
- behave more reliably when several features are strongly correlated.

In scikit-learn, l1_ratio controls the mixture:

- l1_ratio=0: Ridge-like (only L2).
- l1_ratio=1: Lasso-like (only L1).
- 0 < l1_ratio < 1: a mixture of L1 and L2.

**Interview answer:** Elastic Net is useful when the model has many features, possible overfitting, and groups of related features.

## 6. Ridge vs Lasso vs Elastic Net

| Method | Penalty | Main effect | Best memory clue |
|---|---|---|---|
| Ridge | L2: squared weights | Shrinks weights; usually keeps all features | Reduces overfitting |
| Lasso | L1: absolute weights | Shrinks weights; some can become zero | Feature selection |
| Elastic Net | L1 + L2 | Shrinks weights and can select features | Combination of both |

### Quick choice

- Mainly worried about overfitting? Start with **Ridge**.
- Need a small set of selected features? Try **Lasso**.
- Have many related features and want both benefits? Try **Elastic Net**.

These are starting points. Use validation or cross-validation to choose the final model and alpha.

## 7. Small Python example

This example creates a simple dataset. The target mainly depends on useful_1 and useful_2; noise_feature is deliberately not useful. We standardize first because regularization compares coefficient sizes.

In [ ]:
import pandas as pd
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(10)
n_rows = 120

X = pd.DataFrame({
    "useful_1": rng.normal(size=n_rows),
    "useful_2": rng.normal(size=n_rows),
    "noise_feature": rng.normal(size=n_rows),
})
y = 4 * X["useful_1"] - 2 * X["useful_2"] + rng.normal(0, 1, n_rows)

models = {
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Lasso": make_pipeline(StandardScaler(), Lasso(alpha=0.15, max_iter=20_000)),
    "Elastic Net": make_pipeline(StandardScaler(), ElasticNet(alpha=0.15, l1_ratio=0.5, max_iter=20_000)),
}

coefficient_rows = []
for name, model in models.items():
    model.fit(X, y)
    coefficient_rows.append(model[-1].coef_)

coefficient_table = pd.DataFrame(
    coefficient_rows,
    index=models.keys(),
    columns=X.columns,
).round(3)

coefficient_table

### What should you notice?

- useful_1 and useful_2 usually keep larger weights because they help predict y.
- Ridge normally keeps a small non-zero weight for noise_feature.
- Lasso or Elastic Net may make noise_feature exactly zero (the exact result depends on alpha and the data).
- The coefficients are comparable because StandardScaler put the features on a similar scale.

A coefficient becoming zero is a model decision for this dataset; it is not proof that a feature is useless in every dataset.

## 8. Visual: what happens when alpha increases?

The following code refits each model with stronger penalties. Moving right means alpha is increasing. The lines show how the learned coefficients shrink.

In [ ]:
alphas = np.logspace(-2, 1.5, 35)
model_builders = {
    "Ridge": lambda a: Ridge(alpha=a),
    "Lasso": lambda a: Lasso(alpha=a, max_iter=20_000),
    "Elastic Net": lambda a: ElasticNet(alpha=a, l1_ratio=0.5, max_iter=20_000),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (name, build_model) in zip(axes, model_builders.items()):
    paths = []
    for alpha in alphas:
        pipeline = make_pipeline(StandardScaler(), build_model(alpha))
        pipeline.fit(X, y)
        paths.append(pipeline[-1].coef_)
    paths = np.array(paths)
    for column_number, feature_name in enumerate(X.columns):
        ax.plot(alphas, paths[:, column_number], label=feature_name)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xscale("log")
    ax.set_title(name)
    ax.set_xlabel("alpha (stronger penalty →)")
    ax.grid(alpha=0.25)

axes[0].set_ylabel("coefficient after scaling")
axes[-1].legend(fontsize=8)
fig.suptitle("Regularization shrinks coefficients as alpha increases")
fig.tight_layout()
plt.show()

### Read this visual in one minute

1. Start on the left: alpha is small, so the model behaves closer to ordinary linear regression.
2. Move right: the penalty becomes stronger and coefficients move toward zero.
3. Ridge lines usually approach zero without touching it.
4. Lasso and Elastic Net lines can touch zero, meaning those features are switched off.

The exact curves change with the dataset. The rule to remember is: **stronger regularization means smaller model weights**.

## 9. Practical rules that prevent mistakes

- **Scale the features first.** Otherwise a large measurement unit can make a coefficient look small or large unfairly.
- **Keep scaling inside a pipeline.** It should learn the scale from training data only.
- **Do not choose alpha from training accuracy alone.** Use validation or cross-validation.
- **Keep a separate test set** for the final honest check.
- **Do not assume zero means universally useless.** It means the fitted model did not need that feature at the selected alpha.

## 10. Final revision card

- **Ridge = L2:** adds squared weights; reduces overfitting by shrinking coefficients.
- **Lasso = L1:** adds absolute weights; can make coefficients exactly zero, so it performs feature selection.
- **Elastic Net:** combines L1 and L2; useful when there are many and/or correlated features.
- **Alpha / lambda:** controls penalty strength. Bigger usually means smaller coefficients.
- **Feature selection:** a coefficient of zero means that feature is not used by the fitted model.
- **Best practice:** standardize, use a pipeline, tune alpha with cross-validation, and evaluate on unseen data.

### One-line interview answer

**Ridge is mainly for reducing overfitting, Lasso is mainly for feature selection, and Elastic Net combines both.**